# Normalized SPECTER Stage 3-4 Rerun

Rerun of the downstream Stage 3/4 experiment using empty-string-centered, unit-normalized SPECTER2 embeddings. The legacy unnormalized-cache outputs from notebook 6 remain preserved as the baseline.

This notebook writes only normalized-SPECTER outputs, reruns the six Stage 3 branches, and can launch the corrected normalized Stage 4 generation branches.


In [1]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

# Match the working Colab setup used by Notebook 2:
# code repo in /content/neurovlm_gnn, Drive used for data and run outputs.
REPO_URL = os.environ.get("NEUROVLM_REPO_URL", "https://github.com/neurovlm/neurovlm.git")
REPO_BRANCH = os.environ.get("NEUROVLM_REPO_BRANCH", "neurovlm_gnn")
REPO_DIR = Path(os.environ.get("NEUROVLM_REPO_DIR", "/content/neurovlm_gnn"))
DRIVE_ROOT = Path(os.environ.get("NEUROVLM_DRIVE_ROOT", "/content/drive/MyDrive/neurovlm"))
INSTALL_DEPENDENCIES = os.environ.get("NEUROVLM_INSTALL_DEPS", "1") == "1"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def run_cmd(cmd, cwd=None, *, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}")
    return result

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git checkout. Set NEUROVLM_REPO_DIR to a clean path "
            "or remove that folder, then rerun this cell."
        )
    run_cmd(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    checkout = run_cmd(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    if checkout.returncode != 0:
        run_cmd(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"])
    run_cmd(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

os.chdir(REPO_DIR)

if INSTALL_DEPENDENCIES:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "nilearn", "nibabel", "huggingface-hub", "safetensors", "adapters", "transformers", "pyarrow", "matplotlib", "pandas", "scikit-learn", "tqdm", "umap-learn"])
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz,notebook,metrics]"])

sys.path.insert(0, str(REPO_DIR / "experiments" / "3dcnn"))
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR))

print("Working directory:", os.getcwd())
print("Repo branch:", run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, check=False).stdout.strip())
print("Drive root:", DRIVE_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

from atlas_free_cnn.pipeline_outputs import (
    create_stage2_stage3_stage4_run_dir,
    git_info,
    write_json,
    write_status_report,
    write_readme_what_to_look_at,
    write_table,
    flatten_stage3_metrics,
    ae_source_metric_columns,
)
from atlas_free_cnn.stage1_selection_integration import IntegrationConfig, integrate_completed_stage1_selection

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git -C /content/neurovlm_gnn fetch origin neurovlm_gnn
$ git -C /content/neurovlm_gnn checkout neurovlm_gnn
Your branch is up to date with 'origin/neurovlm_gnn'.
$ git -C /content/neurovlm_gnn pull --ff-only origin neurovlm_gnn
Already up to date.
$ /usr/bin/python3 -m pip install -q nilearn nibabel huggingface-hub safetensors adapters transformers pyarrow matplotlib pandas scikit-learn tqdm umap-learn
$ /usr/bin/python3 -m pip install -q -e .[viz,notebook,metrics]
Working directory: /content/neurovlm_gnn
$ git branch --show-current
neurovlm_gnn
Repo branch: neurovlm_gnn
Drive root: /content/drive/MyDrive/neurovlm


## Config Switches

Top-level controls for building/downloading the normalized cache, rerunning the six normalized Stage 3 branches, and running the six corrected normalized Stage 4 branches.


In [ ]:
# 6a top-level controls
CREATE_OR_DOWNLOAD_NORMALIZED_CACHE = True
RERUN_STAGE3_NORMALIZED = True
RUN_CORRECTED_STAGE4_NORMALIZED = True
RUN_FRESH_STAGE3_PROJECTOR_CONTROL = False
UPLOAD_RESULTS_TO_DRIVE = True

NORMALIZED_SPECTER_FILENAME = "specter2_stage3_stage4_emptycentered_unitnorm.pt"
NORMALIZED_SPECTER_METADATA_FILENAME = "specter2_stage3_stage4_emptycentered_unitnorm_metadata.json"
NORMALIZED_SPECTER_VALIDATION_FILENAME = "specter2_stage3_stage4_emptycentered_unitnorm_validation.json"
TEXT_EMBEDDING_PREPROCESSING = "specter2_empty_string_centered_unitnorm"
LEGACY_STAGE4_BASELINES = {
    "mixed_stage1a_on_pubmed": 0.705977,
    "mixed_to_pubmed_stage1b_on_pubmed": 0.838719,
    "mixed_stage1a_on_nilearn": 0.640522,
    "mixed_to_nilearn_stage1b_on_nilearn": 0.610559,
    "mixed_stage1a_on_neurovault": 0.904556,
    "mixed_to_neurovault_stage1b_on_neurovault": 0.885624,
}


In [2]:
def split_dir_has_jsonl(path: Path) -> bool:
    return all((path / name).exists() for name in ["train.jsonl", "val.jsonl", "test.jsonl"])


HF_DATASET_REPO = os.environ.get("NEUROVLM_ATLAS_FREE_HF_REPO", "neurovlm/atlas_free_cnn_dataset")
LOCAL_UNIFIED_CACHE_DIR = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild"
LOCAL_SPLIT_DIR = LOCAL_UNIFIED_CACHE_DIR / "splits"
LOCAL_PACK_DIR = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/hf_atlas_free_cnn_rebuild"


def hf_download_first_available(filenames, local_dir: Path) -> Path:
    from huggingface_hub import hf_hub_download
    local_dir.mkdir(parents=True, exist_ok=True)
    errors = []
    for filename in filenames:
        try:
            path = hf_hub_download(
                repo_id=HF_DATASET_REPO,
                repo_type="dataset",
                filename=filename,
                local_dir=str(local_dir),
                local_dir_use_symlinks=False,
            )
            return Path(path)
        except Exception as exc:
            errors.append(f"{filename}: {exc}")
    raise FileNotFoundError("Could not download any candidate from HF:\n" + "\n".join(errors))


def ensure_hf_unified_splits() -> Path:
    print(f"Downloading atlas-free CNN split JSONLs from Hugging Face: {HF_DATASET_REPO}")
    LOCAL_SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    for split in ["train", "val", "test"]:
        downloaded = hf_download_first_available(
            [f"{split}.jsonl", f"splits/{split}.jsonl", f"unified_jsonl_rebuild/splits/{split}.jsonl"],
            LOCAL_UNIFIED_CACHE_DIR,
        )
        target = LOCAL_SPLIT_DIR / f"{split}.jsonl"
        if downloaded.resolve() != target.resolve():
            shutil.copy2(downloaded, target)
    for name in ["train_map_ids.json", "val_map_ids.json", "test_map_ids.json"]:
        try:
            downloaded = hf_download_first_available(
                [name, f"splits/{name}", f"unified_jsonl_rebuild/splits/{name}"],
                LOCAL_UNIFIED_CACHE_DIR,
            )
            target = LOCAL_SPLIT_DIR / name
            if downloaded.resolve() != target.resolve():
                shutil.copy2(downloaded, target)
        except Exception as exc:
            print(f"Optional split sidecar not downloaded ({name}): {exc}")
    try:
        downloaded_volume = hf_download_first_available(
            ["atlas_free_cnn_volumes.pt", "hf_atlas_free_cnn/atlas_free_cnn_volumes.pt", "hf_atlas_free_cnn_rebuild/atlas_free_cnn_volumes.pt"],
            LOCAL_PACK_DIR,
        )
        target_volume = LOCAL_PACK_DIR / "atlas_free_cnn_volumes.pt"
        if downloaded_volume.resolve() != target_volume.resolve():
            try:
                if target_volume.exists() or target_volume.is_symlink():
                    target_volume.unlink()
                os.symlink(downloaded_volume, target_volume)
            except Exception:
                shutil.copy2(downloaded_volume, target_volume)
        print("Volume tensor available at:", target_volume)
    except Exception as exc:
        print("WARNING: split JSONLs downloaded, but volume tensor was not prepared:", exc)
        print("Training will fail unless tensor_path values inside JSONL resolve to an accessible tensor file.")
    return LOCAL_SPLIT_DIR


def discover_unified_split_dir() -> Path:
    from atlas_free_cnn.notebook_utils import discover_unified_split_dir as _discover_unified_split_dir
    return _discover_unified_split_dir(
        repo_dir=REPO_DIR,
        drive_root=DRIVE_ROOT,
        dataset_repo=HF_DATASET_REPO,
        local_unified_cache_dir=LOCAL_UNIFIED_CACHE_DIR,
        local_split_dir=LOCAL_SPLIT_DIR,
        local_pack_dir=LOCAL_PACK_DIR,
    )
    override = os.environ.get("NEUROVLM_UNIFIED_SPLIT_DIR", "").strip()
    if override:
        override_path = Path(override).expanduser()
        if not split_dir_has_jsonl(override_path):
            raise FileNotFoundError(
                "NEUROVLM_UNIFIED_SPLIT_DIR was set, but it does not contain train.jsonl, val.jsonl, and test.jsonl: "
                f"{override_path}"
            )
        return override_path
    try:
        hf_split_dir = ensure_hf_unified_splits()
        if split_dir_has_jsonl(hf_split_dir):
            return hf_split_dir
    except Exception as exc:
        hf_error = exc
    else:
        hf_error = None
    raise FileNotFoundError(
        "Could not download unified split JSONLs from Hugging Face. Expected root files train.jsonl, val.jsonl, and test.jsonl.\n"
        f"HF dataset repo tried: {HF_DATASET_REPO}\n"
        f"HF fallback error: {hf_error}\n\n"
        "Set NEUROVLM_UNIFIED_SPLIT_DIR only if you need an explicit override."
    )

UNIFIED_SPLIT_DIR = discover_unified_split_dir()
TRAIN_JSONL = str(UNIFIED_SPLIT_DIR / "train.jsonl")
VAL_JSONL = str(UNIFIED_SPLIT_DIR / "val.jsonl")
TEST_JSONL = str(UNIFIED_SPLIT_DIR / "test.jsonl")
print("Unified split dir:", UNIFIED_SPLIT_DIR)
print("Train JSONL:", TRAIN_JSONL)

RUN_MODE = "downstream_only"
DATA_MODE = "mixed"
RERUN_STAGE1_CHECKPOINT_EVALUATION = False

# Required: explicit selected checkpoint paths. By default this uses the completed
# Stage 1A pretraining run for the mixed baseline and the completed Stage 1B
# fine-tuning run for PubMed/Nilearn/NeuroVault. Override individual checkpoint
# paths with NEUROVLM_*_AE_CKPT if your Drive layout differs.
AE_PRETRAINED_RUN_ROOT_VALUE = os.environ.get("NEUROVLM_AE_PRETRAINED_RUN_ROOT", "").strip()
AE_FINETUNED_RUN_ROOT_VALUE = os.environ.get("NEUROVLM_AE_FINETUNED_RUN_ROOT", "").strip()
AE_PRETRAINED_RUN_ROOT = Path(AE_PRETRAINED_RUN_ROOT_VALUE).expanduser() if AE_PRETRAINED_RUN_ROOT_VALUE else DRIVE_ROOT / "runs_atlas_free_cnn_ae_ablation/ae_ablation_20260623_165729"
AE_FINETUNED_RUN_ROOT = Path(AE_FINETUNED_RUN_ROOT_VALUE).expanduser() if AE_FINETUNED_RUN_ROOT_VALUE else DRIVE_ROOT / "runs_atlas_free_cnn_ae_ablation/ae_ablation_20260624_190738"

CONFIGURED_SELECTED_AE_CHECKPOINTS = {
    "mixed_stage1a": {
        "path": os.environ.get(
            "NEUROVLM_MIXED_STAGE1A_AE_CKPT",
            str(AE_PRETRAINED_RUN_ROOT / "01_stage1_ae_pretraining/mixed_baseline_raw_mse/checkpoints/best_top1_dice.pt"),
        ),
        "stage": "stage1a",
        "training_domain": "mixed",
        "checkpoint_name": "best_top1_dice.pt",
        "selection_reason": "held_out_multi_source_rank_1",
        "evaluation_status": "completed",
    },
    "mixed_to_pubmed_stage1b": {
        "path": os.environ.get(
            "NEUROVLM_PUBMED_STAGE1B_AE_CKPT",
            str(AE_FINETUNED_RUN_ROOT / "02_stage1b_ae_finetuning/pubmed/checkpoints/best_top1_dice.pt"),
        ),
        "stage": "stage1b",
        "training_domain": "pubmed",
        "checkpoint_name": "best_top1_dice.pt",
        "selection_reason": "held_out_domain_rank_1",
        "evaluation_status": "completed",
    },
    "mixed_to_nilearn_stage1b": {
        "path": os.environ.get(
            "NEUROVLM_NILEARN_STAGE1B_AE_CKPT",
            str(AE_FINETUNED_RUN_ROOT / "02_stage1b_ae_finetuning/nilearn/checkpoints/best_val_loss.pt"),
        ),
        "stage": "stage1b",
        "training_domain": "nilearn",
        "checkpoint_name": "best_val_loss.pt",
        "selection_reason": "held_out_top5_dice_rank_1",
        "evaluation_status": "completed",
    },
    "mixed_to_neurovault_stage1b": {
        "path": os.environ.get(
            "NEUROVLM_NEUROVAULT_STAGE1B_AE_CKPT",
            str(AE_FINETUNED_RUN_ROOT / "02_stage1b_ae_finetuning/neurovault/checkpoints/best_top5_dice.pt"),
        ),
        "stage": "stage1b",
        "training_domain": "neurovault",
        "checkpoint_name": "best_top5_dice.pt",
        "selection_reason": "held_out_top5_dice_rank_1",
        "evaluation_status": "completed",
    },
}

# Optional provenance-only notebook-7 evaluation output folders. Leave blank if unavailable.
STAGE1A_EVALUATION_DIR_VALUE = os.environ.get("NEUROVLM_STAGE1A_EVALUATION_DIR", "").strip()
STAGE1B_EVALUATION_DIR_VALUE = os.environ.get("NEUROVLM_STAGE1B_EVALUATION_DIR", "").strip()
STAGE1A_EVALUATION_DIR = Path(STAGE1A_EVALUATION_DIR_VALUE).expanduser() if STAGE1A_EVALUATION_DIR_VALUE else None
STAGE1B_EVALUATION_DIR = Path(STAGE1B_EVALUATION_DIR_VALUE).expanduser() if STAGE1B_EVALUATION_DIR_VALUE else None

RUN_STAGE1_SELECTION_INTEGRATION = True
RERUN_STAGE3 = os.environ.get("NEUROVLM_RERUN_STAGE3_NORMALIZED", "1" if RERUN_STAGE3_NORMALIZED else "0") == "1"
RUN_STAGE3_CONTRASTIVE = RERUN_STAGE3
RUN_STAGE4_TEXT_TO_BRAIN = os.environ.get("NEUROVLM_RUN_CORRECTED_STAGE4_NORMALIZED", "1" if RUN_CORRECTED_STAGE4_NORMALIZED else "0") == "1"
RUN_STAGE5_GENERATION_EVAL = True
LEGACY_CONTRASTIVE_INITIALIZED_STAGE4 = False
COMPLETED_STAGE3_RUN_ROOT_VALUE = os.environ.get("NEUROVLM_COMPLETED_STAGE3_RUN_ROOT", "").strip()
DEFAULT_COMPLETED_STAGE3_RUN_ROOT = DRIVE_ROOT / "runs_stage2_stage3_stage4/stage2_stage3_stage4_20260625_210801"
COMPLETED_STAGE3_RUN_ROOT = Path(COMPLETED_STAGE3_RUN_ROOT_VALUE).expanduser() if COMPLETED_STAGE3_RUN_ROOT_VALUE else DEFAULT_COMPLETED_STAGE3_RUN_ROOT
STAGE4_TEXT_EMBEDDING_CACHE_VALUE = os.environ.get("NEUROVLM_STAGE4_TEXT_EMBEDDING_CACHE", "").strip()
STAGE4_TEXT_EMBEDDING_CACHE = Path(STAGE4_TEXT_EMBEDDING_CACHE_VALUE).expanduser() if STAGE4_TEXT_EMBEDDING_CACHE_VALUE else None
STAGE4_TEXT_EMBEDDING_CACHE_FILENAME = os.environ.get("NEUROVLM_STAGE4_TEXT_EMBEDDING_CACHE_FILENAME", "").strip()

NUM_WORKERS = int(os.environ.get("NEUROVLM_NUM_WORKERS", "4" if IN_COLAB else "0"))
EVAL_NUM_WORKERS = int(os.environ.get("NEUROVLM_EVAL_NUM_WORKERS", str(NUM_WORKERS)))
PREFETCH_FACTOR = int(os.environ.get("NEUROVLM_PREFETCH_FACTOR", "4"))
METRICS_DEVICE = os.environ.get("NEUROVLM_METRICS_DEVICE", "cuda")

BASE_OUTPUT_DIR = DRIVE_ROOT if "DRIVE_ROOT" in globals() and UPLOAD_RESULTS_TO_DRIVE else Path(".")
STAGE1_SELECTION_INTEGRATION_OUTPUT_ROOT = DRIVE_ROOT / "runs_stage1_selection_integration_6a_normalized_specter" if "DRIVE_ROOT" in globals() else Path("runs_stage1_selection_integration_6a_normalized_specter")


Unified split dir: /content/neurovlm_gnn/experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits
Train JSONL: /content/neurovlm_gnn/experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits/train.jsonl


In [3]:
paths = create_stage2_stage3_stage4_run_dir(BASE_OUTPUT_DIR, prefix="6a_results")
RUN_DIR = Path(paths["run_dir"])
print(f"Run directory: {RUN_DIR}")

metadata_dir = Path(paths["metadata"])
metadata_dir.mkdir(parents=True, exist_ok=True)
write_json(metadata_dir / "run_config.json", {
    "RUN_MODE": RUN_MODE,
    "DATA_MODE": DATA_MODE,
    "AE_PRETRAINED_RUN_ROOT": str(AE_PRETRAINED_RUN_ROOT),
    "AE_FINETUNED_RUN_ROOT": str(AE_FINETUNED_RUN_ROOT),
    "CONFIGURED_SELECTED_AE_CHECKPOINTS": CONFIGURED_SELECTED_AE_CHECKPOINTS,
    "STAGE1A_EVALUATION_DIR": str(STAGE1A_EVALUATION_DIR or ""),
    "STAGE1B_EVALUATION_DIR": str(STAGE1B_EVALUATION_DIR or ""),
    "RERUN_STAGE1_CHECKPOINT_EVALUATION": RERUN_STAGE1_CHECKPOINT_EVALUATION,
    "RUN_STAGE1_SELECTION_INTEGRATION": RUN_STAGE1_SELECTION_INTEGRATION,
    "RERUN_STAGE3": RERUN_STAGE3,
    "RUN_STAGE3_CONTRASTIVE": RUN_STAGE3_CONTRASTIVE,
    "COMPLETED_STAGE3_RUN_ROOT": str(COMPLETED_STAGE3_RUN_ROOT or ""),
    "RUN_STAGE4_TEXT_TO_BRAIN": RUN_STAGE4_TEXT_TO_BRAIN,
    "STAGE4_TEXT_EMBEDDING_CACHE": str(STAGE4_TEXT_EMBEDDING_CACHE or ""),
    "STAGE4_TEXT_EMBEDDING_CACHE_FILENAME": STAGE4_TEXT_EMBEDDING_CACHE_FILENAME,
    "NORMALIZED_SPECTER_CACHE": str(globals().get("NORMALIZED_SPECTER_CACHE", "")),
    "TEXT_EMBEDDING_PREPROCESSING": TEXT_EMBEDDING_PREPROCESSING,
    "LEGACY_CONTRASTIVE_INITIALIZED_STAGE4": LEGACY_CONTRASTIVE_INITIALIZED_STAGE4,
    "RUN_STAGE5_GENERATION_EVAL": RUN_STAGE5_GENERATION_EVAL,
})
write_json(metadata_dir / "git_info.json", git_info(REPO_DIR))
(metadata_dir / "environment.txt").write_text(sys.version)


Run directory: /content/drive/MyDrive/neurovlm/runs_stage2_stage3_stage4/stage2_stage3_stage4_20260625_171049


50

## Stage 1 Selection Integration: Validate Explicit Checkpoint Paths

In [4]:
SELECTED_AE_CHECKPOINTS = {}
STAGE2_STAGE3_STAGE4_INPUT_MANIFEST = None
STAGE1_SELECTION_INTEGRATION_DIR = None
stage1_selection_status = "not_requested"

if RUN_STAGE1_SELECTION_INTEGRATION:
    integration_result = integrate_completed_stage1_selection(
        IntegrationConfig(
            output_root=STAGE1_SELECTION_INTEGRATION_OUTPUT_ROOT,
            selected_checkpoints=CONFIGURED_SELECTED_AE_CHECKPOINTS,
            stage1a_evaluation_dir=STAGE1A_EVALUATION_DIR,
            stage1b_evaluation_dir=STAGE1B_EVALUATION_DIR,
            rerun_stage1_checkpoint_evaluation=RERUN_STAGE1_CHECKPOINT_EVALUATION,
        )
    )
    stage1_selection_status = integration_result["status"]
    STAGE1_SELECTION_INTEGRATION_DIR = Path(integration_result["output_dir"])
    STAGE2_STAGE3_STAGE4_INPUT_MANIFEST = STAGE1_SELECTION_INTEGRATION_DIR / "03_downstream_usage/stage2_stage3_stage4_input_manifest.json"
    downstream_manifest = json.loads(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST.read_text())
    SELECTED_AE_CHECKPOINTS = downstream_manifest["selected_ae_checkpoints"]
    write_json(metadata_dir / "stage1_selection_integration_result.json", integration_result)
    write_json(metadata_dir / "stage2_stage3_stage4_input_manifest_pointer.json", {"path": str(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST)})
    validation_report = STAGE1_SELECTION_INTEGRATION_DIR / "02_selected_checkpoint_registry/selected_checkpoint_validation.json"
    if stage1_selection_status not in {"completed", "completed_with_warnings"}:
        print("Stage 1 checkpoint validation failed:", stage1_selection_status)
        print("Validation report:", validation_report)
        for row in integration_result.get("blocking_checkpoints", []):
            print(f"- {row.get('key')}: {row.get('status')} -> {row.get('checkpoint_path')}")
            if row.get("warnings"):
                print("  warnings:", row.get("warnings"))
        raise RuntimeError(
            f"Stage 1 selection integration failed: {stage1_selection_status}. "
            f"Set NEUROVLM_AE_CHECKPOINT_ROOT or the individual NEUROVLM_*_AE_CKPT paths. "
            f"See {validation_report}"
        )
    print("Stage 1 checkpoint validation:", stage1_selection_status)
    print("Integration output:", STAGE1_SELECTION_INTEGRATION_DIR)
    print("Downstream manifest:", STAGE2_STAGE3_STAGE4_INPUT_MANIFEST)
else:
    print("Stage 1 selection integration not requested")

Stage 1 checkpoint validation: completed
Integration output: /content/drive/MyDrive/neurovlm/runs_stage1_selection_integration/stage1_selection_integration_20260625_171049
Downstream manifest: /content/drive/MyDrive/neurovlm/runs_stage1_selection_integration/stage1_selection_integration_20260625_171049/03_downstream_usage/stage2_stage3_stage4_input_manifest.json


## Preserved Stage 1 Evaluation Tables

In [5]:
if STAGE1_SELECTION_INTEGRATION_DIR:
    for path in [
        STAGE1_SELECTION_INTEGRATION_DIR / "01_existing_evaluation_tables/mixed_stage1a_checkpoint_selection.csv",
        STAGE1_SELECTION_INTEGRATION_DIR / "01_existing_evaluation_tables/pubmed_stage1b_checkpoint_selection.csv",
        STAGE1_SELECTION_INTEGRATION_DIR / "01_existing_evaluation_tables/nilearn_stage1b_checkpoint_selection.csv",
        STAGE1_SELECTION_INTEGRATION_DIR / "01_existing_evaluation_tables/neurovault_stage1b_checkpoint_selection.csv",
        STAGE1_SELECTION_INTEGRATION_DIR / "02_selected_checkpoint_registry/selected_ae_checkpoints_for_stage2_stage3_stage4.json",
        STAGE1_SELECTION_INTEGRATION_DIR / "02_selected_checkpoint_registry/selected_checkpoint_validation.json",
        STAGE1_SELECTION_INTEGRATION_DIR / "03_downstream_usage/six_run_ae_assignment.csv",
    ]:
        print(path, "exists=", path.exists())
else:
    print("No Stage 1 selection integration output available")

/content/drive/MyDrive/neurovlm/runs_stage1_selection_integration/stage1_selection_integration_20260625_171049/01_existing_evaluation_tables/mixed_stage1a_checkpoint_selection.csv exists= False
/content/drive/MyDrive/neurovlm/runs_stage1_selection_integration/stage1_selection_integration_20260625_171049/01_existing_evaluation_tables/pubmed_stage1b_checkpoint_selection.csv exists= False
/content/drive/MyDrive/neurovlm/runs_stage1_selection_integration/stage1_selection_integration_20260625_171049/01_existing_evaluation_tables/nilearn_stage1b_checkpoint_selection.csv exists= False
/content/drive/MyDrive/neurovlm/runs_stage1_selection_integration/stage1_selection_integration_20260625_171049/01_existing_evaluation_tables/neurovault_stage1b_checkpoint_selection.csv exists= False
/content/drive/MyDrive/neurovlm/runs_stage1_selection_integration/stage1_selection_integration_20260625_171049/02_selected_checkpoint_registry/selected_ae_checkpoints_for_stage2_stage3_stage4.json exists= True
/conte

## Part 1: Build or Download Normalized SPECTER2 Cache

This cell lists the Hugging Face repository, builds or downloads `text_embeddings/specter2_stage3_stage4_emptycentered_unitnorm.pt`, validates 768-dimensional unit norms, and fails before training if the wrong cache is loaded.


In [ ]:
import torch

def sha256_file(path: Path) -> str:
    import hashlib
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def normalized_cache_paths() -> dict:
    local_dir = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/text_embeddings"
    return {
        "pt": local_dir / NORMALIZED_SPECTER_FILENAME,
        "metadata": local_dir / NORMALIZED_SPECTER_METADATA_FILENAME,
        "validation": local_dir / NORMALIZED_SPECTER_VALIDATION_FILENAME,
        "index": local_dir / "specter2_stage3_stage4_emptycentered_unitnorm_index.csv",
    }


def print_relevant_hf_files() -> list[str]:
    from huggingface_hub import HfApi
    files = HfApi().list_repo_files(repo_id=HF_DATASET_REPO, repo_type="dataset")
    print("Potentially relevant Hugging Face files")
    for name in files:
        lower = name.lower()
        if "specter" in lower or lower.endswith(".jsonl") or "manifest" in lower or lower.endswith(".parquet") or "metadata" in lower:
            print("-", name)
    return files


def download_normalized_cache_if_available(paths: dict) -> bool:
    local_dir = paths["pt"].parent
    candidates = [
        f"text_embeddings/{NORMALIZED_SPECTER_FILENAME}",
        NORMALIZED_SPECTER_FILENAME,
    ]
    try:
        downloaded = hf_download_first_available(candidates, local_dir)
    except Exception as exc:
        print("Normalized cache was not available on HF yet:", exc)
        return False
    paths["pt"].parent.mkdir(parents=True, exist_ok=True)
    if downloaded.resolve() != paths["pt"].resolve():
        shutil.copy2(downloaded, paths["pt"])
    for filename, key in [
        (NORMALIZED_SPECTER_METADATA_FILENAME, "metadata"),
        (NORMALIZED_SPECTER_VALIDATION_FILENAME, "validation"),
        ("specter2_stage3_stage4_emptycentered_unitnorm_index.csv", "index"),
    ]:
        try:
            sidecar = hf_download_first_available([f"text_embeddings/{filename}", filename], local_dir)
            if sidecar.resolve() != paths[key].resolve():
                shutil.copy2(sidecar, paths[key])
        except Exception as exc:
            print(f"Optional normalized sidecar not downloaded ({filename}): {exc}")
    return paths["pt"].exists()


def build_normalized_cache(paths: dict) -> None:
    cmd = [
        sys.executable,
        "experiments/3dcnn/atlas_free_cnn/data_building/build_normalized_specter2_cache.py",
        "--repo-id", HF_DATASET_REPO,
        "--repo-dir", str(REPO_DIR),
        "--local-dir", str(REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/hf_normalized_specter2"),
        "--output-dir", str(paths["pt"].parent),
        "--output-basename", "specter2_stage3_stage4_emptycentered_unitnorm",
    ]
    print("Building normalized SPECTER2 cache")
    print(" ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"Normalized SPECTER2 cache build failed with exit code {result.returncode}")


def required_primary_text_ids() -> set[str]:
    ids = set()
    for split_path in [Path(TRAIN_JSONL), Path(VAL_JSONL), Path(TEST_JSONL)]:
        with split_path.open() as f:
            for line in f:
                if not line.strip():
                    continue
                row = json.loads(line)
                positives = row.get("positive_texts") or []
                if positives:
                    pos = positives[0]
                    ids.add(str(pos.get("text_id") or pos.get("id") or ""))
    ids.discard("")
    return ids


def validate_normalized_cache(path: Path) -> dict:
    payload = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(payload, dict) or "embeddings" not in payload:
        raise RuntimeError("Wrong cache loaded: expected structured normalized cache with an embeddings tensor")
    embeddings = payload["embeddings"].float()
    if embeddings.ndim != 2 or embeddings.shape[1] != 768:
        raise RuntimeError(f"Wrong cache loaded: expected N x 768, got {tuple(embeddings.shape)}")
    if not torch.isfinite(embeddings).all():
        raise RuntimeError("Wrong cache loaded: normalized vectors contain NaNs or infinities")
    norms = embeddings.norm(dim=1)
    stats = {
        "n": int(embeddings.shape[0]),
        "dim": int(embeddings.shape[1]),
        "norm_mean": float(norms.mean().item()),
        "norm_std": float(norms.std(unbiased=False).item()),
        "norm_min": float(norms.min().item()),
        "norm_max": float(norms.max().item()),
        "fraction_within_1e_4": float((norms.sub(1).abs() <= 1e-4).float().mean().item()),
        "fraction_within_1e_3": float((norms.sub(1).abs() <= 1e-3).float().mean().item()),
        "sha256": sha256_file(path),
    }
    if stats["fraction_within_1e_3"] < 0.999:
        raise RuntimeError(f"Wrong cache loaded: vectors are not approximately unit-normalized: {stats}")
    text_ids = {str(v) for v in payload.get("text_ids", [])}
    required_ids = required_primary_text_ids()
    missing = sorted(required_ids - text_ids)
    if missing:
        raise RuntimeError(f"Normalized cache missing {len(missing)} required manifest text IDs; first={missing[:5]}")
    metadata = payload.get("metadata", {})
    if metadata.get("text_embedding_preprocessing") not in {TEXT_EMBEDDING_PREPROCESSING, ""}:
        raise RuntimeError(f"Wrong preprocessing convention: {metadata.get('text_embedding_preprocessing')}")
    print("Normalized SPECTER2 cache stats")
    print(json.dumps(stats, indent=2))
    return {"stats": stats, "metadata": metadata}


if CREATE_OR_DOWNLOAD_NORMALIZED_CACHE:
    print_relevant_hf_files()

NORMALIZED_PATHS = normalized_cache_paths()
if not NORMALIZED_PATHS["pt"].exists():
    downloaded = download_normalized_cache_if_available(NORMALIZED_PATHS)
    if not downloaded and CREATE_OR_DOWNLOAD_NORMALIZED_CACHE:
        build_normalized_cache(NORMALIZED_PATHS)

NORMALIZED_SPECTER_CACHE = Path(os.environ.get("NEUROVLM_NORMALIZED_SPECTER_CACHE", str(NORMALIZED_PATHS["pt"]))).expanduser()
if not NORMALIZED_SPECTER_CACHE.exists():
    raise FileNotFoundError(f"Normalized SPECTER2 cache is missing: {NORMALIZED_SPECTER_CACHE}")
NORMALIZED_CACHE_AUDIT = validate_normalized_cache(NORMALIZED_SPECTER_CACHE)
TEXT_EMBEDDING_CACHE = NORMALIZED_SPECTER_CACHE
STAGE4_TEXT_EMBEDDING_CACHE = NORMALIZED_SPECTER_CACHE
print("NORMALIZED_SPECTER_CACHE =", NORMALIZED_SPECTER_CACHE)
print("TEXT_EMBEDDING_PREPROCESSING =", TEXT_EMBEDDING_PREPROCESSING)


## Stage 2/3: Six Controlled Encoder Initialization Runs

In [6]:
TEXT_EMBEDDING_CACHE = NORMALIZED_SPECTER_CACHE
print("Normalized Stage 3 text embedding cache:", TEXT_EMBEDDING_CACHE)
print("Text preprocessing:", TEXT_EMBEDDING_PREPROCESSING)

def copy_stage3_checkpoint_aliases(ckpt_dir: Path) -> None:
    best = ckpt_dir / "best_ale_cnn.pt"
    last = ckpt_dir / "last_ale_cnn.pt"
    if best.exists():
        shutil.copy2(best, ckpt_dir / "best_val_normalized_recall_auc.pt")
    if last.exists():
        shutil.copy2(last, ckpt_dir / "last.pt")

if RUN_STAGE3_CONTRASTIVE:
    if not STAGE2_STAGE3_STAGE4_INPUT_MANIFEST or not Path(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST).exists():
        raise RuntimeError("Stage 2/3 requires the validated Stage 1 selected-checkpoint manifest")
    downstream_manifest = json.loads(Path(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST).read_text())
    domain_dirs = {"pubmed": "01_pubmed", "nilearn": "02_nilearn", "neurovault": "03_neurovault"}
    specialized_dirs = {
        "pubmed": "specialized_mixed_to_pubmed",
        "nilearn": "specialized_mixed_to_nilearn",
        "neurovault": "specialized_mixed_to_neurovault",
    }
    for run in downstream_manifest["six_stage2_stage3_stage4_runs"]:
        domain = run["domain"]
        branch = "baseline_mixed_stage1a" if run["type"] == "baseline" else specialized_dirs[domain]
        stage3_dir = RUN_DIR / domain_dirs[domain] / branch / "stage3_normalized_specter"
        ckpt_dir = stage3_dir / "checkpoints"
        complete_marker = stage3_dir / "NORMALIZED_STAGE3_COMPLETE.json"
        if complete_marker.exists():
            print("Skipping complete normalized Stage 3 run:", stage3_dir)
            copy_stage3_checkpoint_aliases(ckpt_dir)
            continue
        ae_entry = downstream_manifest["selected_ae_checkpoints"][run["ae_registry_key"]]
        checkpoint_name = Path(ae_entry["checkpoint_name"]).stem
        cmd = [
            sys.executable, "experiments/3dcnn/atlas_free_cnn/training/train_ale_cnn.py",
            "--mode", "atlas_free",
            "--model", "ale_3dcnn",
            "--train-jsonl", TRAIN_JSONL,
            "--val-jsonl", VAL_JSONL,
            "--test-jsonl", TEST_JSONL,
            "--text-embedding-cache", str(TEXT_EMBEDDING_CACHE),
            "--domain", domain,
            "--target-shape", "36,45,38",
            "--epochs", "150",
            "--early-stopping-patience", "25",
            "--val-interval", "1",
            "--batch-size", "512",
            "--base-channels", "64",
            "--num-blocks", "4",
            "--out-dim", "384",
            "--lr-cnn", "0.0001",
            "--lr-proj", "0.00001",
            "--weight-decay", "0.0001",
            "--warmup-epochs", "5",
            "--temperature", "0.07",
            "--dropout", "0.1",
            "--norm", "group",
            "--pooling", "max",
            "--encoder-init", "autoencoder_pretrained",
            "--ae-ckpt-path", str(ae_entry["path"]),
            "--ae-init-variant", str(run["ae_registry_key"]),
            "--ae-checkpoint-selection", checkpoint_name,
            "--text-proj-init", "pretrained_infonce",
            "--monitor-metric", "paper_recall_curve_auc",
            "--run-dir", str(stage3_dir),
            "--checkpoint-dir", str(ckpt_dir),
        ]
        print("Launching normalized Stage 3", f"{run['run']}_normalized_specter")
        print(" ".join(cmd))
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.returncode != 0:
            if result.stderr:
                print(result.stderr)
            raise RuntimeError(f"Normalized Stage 3 run failed for {run['run']} with exit code {result.returncode}")
        copy_stage3_checkpoint_aliases(ckpt_dir)
        write_json(stage3_dir / "normalized_specter_provenance.json", {
            "run": f"{run['run']}_normalized_specter",
            "normalized_cache": str(TEXT_EMBEDDING_CACHE),
            "normalized_cache_sha256": sha256_file(TEXT_EMBEDDING_CACHE),
            "text_embedding_preprocessing": TEXT_EMBEDDING_PREPROCESSING,
            "primary_metric": "mean bidirectional normalized recall@k curve AUC over k/N",
            "checkpoint_alias": str(ckpt_dir / "best_val_normalized_recall_auc.pt"),
            "text_proj_init": "pretrained_infonce",
            "text_projection_trainable": True,
            "single_positive_policy": "one selected primary text per map",
        })
        write_json(complete_marker, {"status": "complete", "stage3_dir": str(stage3_dir)})
else:
    print("Normalized Stage 3 not requested")

if RUN_FRESH_STAGE3_PROJECTOR_CONTROL:
    print("Fresh-projector control is enabled, but not part of the primary six-run experiment. Add a separate control loop here if needed.")


Launching mixed_stage1a_on_pubmed
/usr/bin/python3 experiments/3dcnn/atlas_free_cnn/training/train_ale_cnn.py --mode atlas_free --model ale_3dcnn --epochs 200 --early-stopping-patience 25 --val-interval 1 --batch-size-auto --batch-size-candidates 512,384,256,192,128,96,64,48,32,16 --base-channels 64 --num-blocks 4 --out-dim 384 --dropout 0.1 --norm group --pooling max --encoder-init autoencoder_pretrained --ae-ckpt-path /content/drive/MyDrive/neurovlm/runs_atlas_free_cnn_ae_ablation/ae_ablation_20260623_165729/01_stage1_ae_pretraining/mixed_baseline_raw_mse/checkpoints/best_top1_dice.pt --ae-init-variant mixed_stage1a --ae-checkpoint-selection best_top1_dice --text-proj-init pretrained_infonce --run-dir /content/drive/MyDrive/neurovlm/runs_stage2_stage3_stage4/stage2_stage3_stage4_20260625_171049/01_pubmed/baseline_mixed_stage1a/stage3 --checkpoint-dir /content/drive/MyDrive/neurovlm/runs_stage2_stage3_stage4/stage2_stage3_stage4_20260625_171049/01_pubmed/baseline_mixed_stage1a/stage3/

# Stage 4: Run All Six Controlled Variants

Stage 4 must run for all six Stage 2/3 variants, not only the Stage 3 winners. The goal is to compare whether domain-specific Stage 1B AE fine-tuning improves both Stage 3 retrieval and Stage 4 text-to-brain generation.

All comparisons are within-domain only. Baseline and specialized runs for the same domain must use identical train, validation, and test splits, text IDs, SPECTER embeddings, batching, optimizer, schedule, random seed where practical, loss, early stopping, checkpoint-selection policy, and evaluation implementation.

## Selected AE Checkpoints

Use the explicit selected-checkpoint registry produced above. Do not infer checkpoints from filename labels, directory order, mtime, the most recent run, or generic aliases.

Required choices:

| Registry key | Checkpoint | Selection reason |
| --- | --- | --- |
| `mixed_stage1a` | `mixed_baseline_raw_mse/checkpoints/best_top1_dice.pt` | `held_out_multi_source_rank_1` |
| `mixed_to_pubmed_stage1b` | `pubmed/checkpoints/best_top1_dice.pt` | `held_out_domain_rank_1` |
| `mixed_to_nilearn_stage1b` | `nilearn/checkpoints/best_val_loss.pt` | `held_out_top5_dice_rank_1` |
| `mixed_to_neurovault_stage1b` | `neurovault/checkpoints/best_top5_dice.pt` | `held_out_top5_dice_rank_1` |

## Six Stage 4 Runs

| Stage 4 run | Domain | Type | AE registry key | Stage 3 source |
| --- | --- | --- | --- | --- |
| `mixed_stage1a_on_pubmed_stage4` | PubMed | baseline | `mixed_stage1a` | `mixed_stage1a_on_pubmed` |
| `mixed_to_pubmed_stage1b_on_pubmed_stage4` | PubMed | specialized | `mixed_to_pubmed_stage1b` | `mixed_to_pubmed_stage1b_on_pubmed` |
| `mixed_stage1a_on_nilearn_stage4` | Nilearn | baseline | `mixed_stage1a` | `mixed_stage1a_on_nilearn` |
| `mixed_to_nilearn_stage1b_on_nilearn_stage4` | Nilearn | specialized | `mixed_to_nilearn_stage1b` | `mixed_to_nilearn_stage1b_on_nilearn` |
| `mixed_stage1a_on_neurovault_stage4` | NeuroVault | baseline | `mixed_stage1a` | `mixed_stage1a_on_neurovault` |
| `mixed_to_neurovault_stage1b_on_neurovault_stage4` | NeuroVault | specialized | `mixed_to_neurovault_stage1b` | `mixed_to_neurovault_stage1b_on_neurovault` |

## Component Matching

Each Stage 4 run must load only matching upstream components:

1. Stage 3 text-side representation/projection from the matching Stage 3 run.
2. AE decoder from the exact AE checkpoint used to initialize that Stage 3 run.
3. Correct domain-specific train, validation, and held-out test splits.
4. A new Stage 4 text-to-latent projection head unique to that run.

Fail before training if a specialized Stage 3 checkpoint is paired with the mixed decoder, a mixed Stage 3 checkpoint is paired with a Stage 1B decoder, components are crossed across domains, a projection head is reused, or a run is evaluated on the wrong domain test split.

## Required Preflight And Provenance

Before each Stage 4 run, write `stage4_component_provenance.json` and `stage4_trainable_parameter_report.json`.

Validate and record:

* AE checkpoint path, filename, stage, training domain, selection reason, epoch, encoder checksum, decoder checksum.
* Stage 3 run/checkpoint path, epoch, selection metric, text projection checksum.
* SPECTER model/version.
* Train/validation/test split fingerprints.
* Decoder, Stage 3 text projection, and Stage 4 projection trainable/frozen status.
* Stage 3 domain equals Stage 4 domain.
* Stage 3 AE initialization path equals the AE checkpoint supplying the decoder.
* Decoder checksum matches the registered AE checkpoint.
* Baseline/specialized split fingerprints match within domain.
* No test example appears in train or validation data.

Dimensional preflight must verify:

```text
text batch -> SPECTER/text embedding -> Stage 3 text projection -> Stage 4 latent [batch, 384] -> decoder output [batch, 1, 36, 45, 38]
```

Primary trainability policy:

* AE decoder frozen.
* Stage 4 text-to-AE-latent projection trainable.
* Stage 3 CNN encoder not updated for generation.
* Stage 3 text projection frozen unless the previously validated Stage 4 recipe trained it; if so, use that same policy for all six runs and log it.
* SPECTER/base embeddings unchanged.

## Checkpointing, Evaluation, And Outputs

For each Stage 4 run, save:

* `best_val_loss.pt`
* `best_val_spatial_corr.pt`
* `best_val_top5_dice.pt`
* `best_val_foreground_mse.pt`
* `last.pt`
* training/validation histories and per-example held-out test generation metrics
* generated maps or compact prediction tensor plus manifest

Do not select final generation checkpoint by validation MSE alone. Prefer validation spatial correlation, top-5 Dice/overlap, foreground reconstruction quality, then validation MSE.

Test split is evaluation-only: no gradients, early stopping, checkpoint selection, or hyperparameter decisions.

## Comparison Files

Create within-domain Stage 4 comparisons:

* `pubmed_stage4_baseline_vs_specialized.csv`
* `nilearn_stage4_baseline_vs_specialized.csv`
* `neurovault_stage4_baseline_vs_specialized.csv`

Also create:

* `all_domain_stage4_comparison.csv`
* `ae_retrieval_generation_comparison.csv`

Primary conclusions must come from within-domain paired comparisons, not raw cross-domain ranking.

## Completion Rule

A Stage 4 run is complete only if saved outputs exist and validate: config, provenance, trainable-parameter report, checkpoint, training/validation histories, held-out generation metrics, nonzero predictions, per-example metrics, and generated-map manifest. The overall pipeline is complete only when all six Stage 4 variants are complete.

In [7]:
def discover_stage3_checkpoint(domain: str, branch: str) -> Path:
    if RERUN_STAGE3:
        candidate = RUN_DIR / domain_dirs[domain] / branch / "stage3_normalized_specter/checkpoints/best_val_normalized_recall_auc.pt"
    else:
        if COMPLETED_STAGE3_RUN_ROOT is None:
            raise RuntimeError(
                "RERUN_STAGE3 is False, so Stage 4 needs explicit completed Stage 3 outputs. "
                "Set NEUROVLM_COMPLETED_STAGE3_RUN_ROOT to the run directory containing 01_pubmed/, 02_nilearn/, and 03_neurovault/."
            )
        candidate = COMPLETED_STAGE3_RUN_ROOT / domain_dirs[domain] / branch / "stage3_normalized_specter/checkpoints/best_val_normalized_recall_auc.pt"
    if not candidate.exists():
        raise FileNotFoundError(f"Missing matching Stage 3 checkpoint: {candidate}")
    return candidate


def discover_stage4_generation_cache() -> Path:
    if NORMALIZED_SPECTER_CACHE is None or not Path(NORMALIZED_SPECTER_CACHE).exists():
        raise FileNotFoundError(f"Normalized SPECTER2 cache does not exist: {NORMALIZED_SPECTER_CACHE}")
    return Path(NORMALIZED_SPECTER_CACHE)



STAGE4_GENERATION_TEXT_CACHE = discover_stage4_generation_cache() if RUN_STAGE4_TEXT_TO_BRAIN else None
if STAGE4_GENERATION_TEXT_CACHE:
    print("Corrected Stage 4 generation cache:", STAGE4_GENERATION_TEXT_CACHE)

if RUN_STAGE4_TEXT_TO_BRAIN:
    if not STAGE2_STAGE3_STAGE4_INPUT_MANIFEST or not Path(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST).exists():
        raise RuntimeError("Stage 4 requires the validated Stage 1 selected-checkpoint manifest")
    downstream_manifest = json.loads(Path(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST).read_text())
    domain_dirs = {"pubmed": "01_pubmed", "nilearn": "02_nilearn", "neurovault": "03_neurovault"}
    specialized_dirs = {
        "pubmed": "specialized_mixed_to_pubmed",
        "nilearn": "specialized_mixed_to_nilearn",
        "neurovault": "specialized_mixed_to_neurovault",
    }
    stage4_epochs = int(os.environ.get("NEUROVLM_STAGE4_EPOCHS", "200"))
    stage4_patience = int(os.environ.get("NEUROVLM_STAGE4_EARLY_STOPPING_PATIENCE", "25"))
    stage4_val_interval = int(os.environ.get("NEUROVLM_STAGE4_VAL_INTERVAL", "1"))
    generation_auc_val_interval = int(os.environ.get("NEUROVLM_GENERATION_AUC_VAL_INTERVAL", "5"))
    stage4_summaries = []
    for run in downstream_manifest["six_stage2_stage3_stage4_runs"]:
        domain = run["domain"]
        branch = "baseline_mixed_stage1a" if run["type"] == "baseline" else specialized_dirs[domain]
        branch_dir = RUN_DIR / domain_dirs[domain] / branch
        stage3_ckpt = discover_stage3_checkpoint(domain, branch)
        stage3_dir = stage3_ckpt.parents[1]
        stage4_dir = branch_dir / "corrected_stage4_normalized_specter"
        ckpt_dir = stage4_dir / "checkpoints"
        config_dir = stage4_dir / "config"
        for rel in ["config", "provenance", "checkpoints", "training_metrics", "validation_metrics", "test_metrics", "per_example", "recall_curves", "generated_maps", "plots"]:
            (stage4_dir / rel).mkdir(parents=True, exist_ok=True)
        ae_entry = downstream_manifest["selected_ae_checkpoints"][run["ae_registry_key"]]
        provenance = {
            "stage4_run": f"{run['run']}_normalized_stage4",
            "domain": domain,
            "type": run["type"],
            "architecture": "generative_text_to_ae_latent",
            "legacy_contrastive_initialized_stage4": False,
            "ae_registry_key": run["ae_registry_key"],
            "ae_checkpoint_path": ae_entry["path"],
            "ae_checkpoint_name": ae_entry["checkpoint_name"],
            "ae_stage": ae_entry["stage"],
            "ae_training_domain": ae_entry["training_domain"],
            "ae_selection_reason": ae_entry["selection_reason"],
            "stage3_dir": str(stage3_dir),
            "stage3_checkpoint": str(stage3_ckpt),
            "stage3_usage": "semantic_validation_and_test_evaluation_only",
            "train_jsonl": TRAIN_JSONL,
            "val_jsonl": VAL_JSONL,
            "test_jsonl": TEST_JSONL,
            "stage4_text_embedding_cache": str(STAGE4_GENERATION_TEXT_CACHE),
            "stage4_text_embedding_cache_sha256": sha256_file(STAGE4_GENERATION_TEXT_CACHE),
            "text_embedding_preprocessing": TEXT_EMBEDDING_PREPROCESSING,
            "single_positive_policy": "one selected primary text per map",
        }
        write_json(stage4_dir / "provenance/stage4_component_provenance.json", provenance)
        write_json(stage4_dir / "stage4_trainable_parameter_report.json", {
            "frozen": ["autoencoder_encoder", "autoencoder_decoder", "stage3_contrastive_projector", "stage3_brain_encoder"],
            "trainable": ["generative_text_to_ae_latent"],
            "forbidden": ["stage3_contrastive_projector_weight_loading", "stage3_contrastive_embedding_decoder_input"],
            "note": "Corrected Stage 4 learns a fresh 768->512->384 projector into the raw frozen AE latent space.",
        })
        cfg = {
            "train_jsonl": TRAIN_JSONL,
            "val_jsonl": VAL_JSONL,
            "test_jsonl": TEST_JSONL,
            "eval_jsonls": {},
            "text_embedding_cache": str(STAGE4_GENERATION_TEXT_CACHE),
            "text_embedding_preprocessing": TEXT_EMBEDDING_PREPROCESSING,
            "autoencoder_checkpoint": ae_entry["path"],
            "stage3_contrastive_checkpoint": str(stage3_ckpt),
            "domain": domain,
            "output_dir": str(stage4_dir),
            "checkpoint_dir": str(ckpt_dir),
            "device": "auto",
            "seed": 42,
            "target_shape": [36, 45, 38],
            "batch_size": int(os.environ.get("NEUROVLM_STAGE4_BATCH_SIZE", "1024")),
            "preflight_batch_size": True,
            "batch_candidates": [4096, 3072, 2048, 1536, 1024, 768, 512, 384, 256, 192, 128, 96, 64],
            "runtime_batch_fallback": True,
            "num_workers": NUM_WORKERS,
            "pin_memory": True,
            "persistent_workers": NUM_WORKERS > 0,
            "prefetch_factor": PREFETCH_FACTOR,
            "epochs": stage4_epochs,
            "early_stopping": True,
            "early_stopping_metric": "val_generation_normalized_auc",
            "early_stopping_mode": "max",
            "early_stopping_patience": stage4_patience,
            "early_stopping_min_delta": 0.0,
            "lr": float(os.environ.get("NEUROVLM_STAGE4_LR", "0.00005")),
            "weight_decay": float(os.environ.get("NEUROVLM_STAGE4_WEIGHT_DECAY", "0.0001")),
            "positive_texts_per_map": 1,
            "text_projection_init": "random",
            "legacy_contrastive_initialized_stage4": False,
            "prediction_activation": "none",
            "amp": True,
            "cudnn_benchmark": True,
            "val_interval": stage4_val_interval,
            "generation_auc_val_interval": generation_auc_val_interval,
            "generation_auc_batch_size": int(os.environ.get("NEUROVLM_GENERATION_AUC_BATCH_SIZE", "512")),
            "compute_train_metrics": True,
            "train_metric_batches": 8,
            "val_metric_batches": 16,
            "metrics_device": METRICS_DEVICE,
            "include_voxel_auroc": False,
            "eval_num_workers": EVAL_NUM_WORKERS,
            "model": {
                "latent_dim": 384,
                "base_channels": 64,
                "num_blocks": 4,
                "encoder_arch": "plain",
                "dropout": 0.1,
                "norm": "group",
                "pooling": "max",
                "blocks_per_stage": 2,
                "use_dilation": False,
                "multi_scale": False,
                "global_context": "none",
            },
            "generative_text_to_ae_latent": {"name": "generative_text_to_ae_latent", "in_dim": 768, "hidden_dim": 512, "latent_dim": 384},
            "weighted_recon": {"type": "mse", "alpha": 0.0, "gamma": 1.0, "normalize_target": True},
            "loss_name": "latent_mse_plus_reconstruction_mse",
            "loss": {"lambda_recon": 1.0, "lambda_latent": 1.0, "lambda_dice": 0.0, "lambda_topk": 0.0, "lambda_corr": 0.0},
            "optional_loss_ablations_supported_not_run_by_default": [
                "latent_mse_only",
                "latent_mse_plus_reconstruction_mse",
                "latent_mse_plus_cosine_plus_reconstruction_mse",
            ],
        }
        config_path = config_dir / "text_to_brain_config.json"
        write_json(config_path, cfg)
        cmd = [sys.executable, "-m", "atlas_free_cnn.training.train_text_to_brain", "--config", str(config_path)]
        env = os.environ.copy()
        env["PYTHONPATH"] = os.pathsep.join([
            str(REPO_DIR / "experiments" / "3dcnn"),
            str(REPO_DIR / "src"),
            str(REPO_DIR),
            env.get("PYTHONPATH", ""),
        ])
        print("Launching corrected Stage 4", f"{run['run']}_normalized_stage4")
        print(" ".join(cmd))
        result = subprocess.run(cmd, text=True, capture_output=True, env=env)
        if result.stdout:
            print(result.stdout)
        if result.returncode != 0:
            if result.stderr:
                print(result.stderr)
            raise RuntimeError(f"Corrected Stage 4 run failed for {run['run']} with exit code {result.returncode}")
        stage4_summaries.append({"run": f"{run['run']}_normalized_stage4", "stage4_dir": str(stage4_dir), "config": str(config_path), "primary_checkpoint": str(ckpt_dir / "best_val_generation_normalized_auc.pt")})
    write_json(Path(paths["stage4"]) / "corrected_stage4_run_manifest.json", stage4_summaries)
else:
    print("Corrected Stage 4 not requested")

if "stage5" not in paths:
    paths["stage5"] = str(RUN_DIR / "04_stage5_generation_eval")
    Path(paths["stage5"]).mkdir(parents=True, exist_ok=True)

if RUN_STAGE5_GENERATION_EVAL:
    print("Corrected Stage 4 semantic/spatial test evaluation is produced by the trainer and notebook 7 diagnostics.")
else:
    print("Stage 5 not requested")


Configure and launch atlas_free_cnn.training.train_text_to_brain here; outputs should go under /content/drive/MyDrive/neurovlm/runs_stage2_stage3_stage4/stage2_stage3_stage4_20260625_171049/03_stage4_text_to_brain_generation


KeyError: 'stage5'

## Final Status and Comparison Files

In [ ]:
stage_status = write_status_report(RUN_DIR, {
    "stage1_selection_integration": stage1_selection_status,
    "stage3_rerun": RUN_STAGE3_CONTRASTIVE,
    "stage4": RUN_STAGE4_TEXT_TO_BRAIN,
    "stage5": RUN_STAGE5_GENERATION_EVAL,
})

if STAGE1_SELECTION_INTEGRATION_DIR:
    assignment_csv = STAGE1_SELECTION_INTEGRATION_DIR / "03_downstream_usage/six_run_ae_assignment.csv"
    final_assignment = Path(paths["final"]) / "six_run_ae_assignment.csv"
    if assignment_csv.exists():
        shutil.copy2(assignment_csv, final_assignment)
    write_table(Path(paths["final"]) / "final_summary_table.csv", [{"stage": s["stage"], "status": s["status"]} for s in stage_status])
    write_json(Path(paths["final"]) / "stage1_selection_integration_pointer.json", {
        "integration_dir": str(STAGE1_SELECTION_INTEGRATION_DIR),
        "downstream_manifest": str(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST),
    })
else:
    write_table(Path(paths["final"]) / "final_summary_table.csv", [{"stage": s["stage"], "status": s["status"]} for s in stage_status])

audit_rows = []
try:
    manifest_for_audit = json.loads(Path(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST).read_text()) if STAGE2_STAGE3_STAGE4_INPUT_MANIFEST else {"six_stage2_stage3_stage4_runs": []}
    domain_dirs = {"pubmed": "01_pubmed", "nilearn": "02_nilearn", "neurovault": "03_neurovault"}
    specialized_dirs = {"pubmed": "specialized_mixed_to_pubmed", "nilearn": "specialized_mixed_to_nilearn", "neurovault": "specialized_mixed_to_neurovault"}
    pair_fingerprints = {}
    for run in manifest_for_audit["six_stage2_stage3_stage4_runs"]:
        domain = run["domain"]
        branch = "baseline_mixed_stage1a" if run["type"] == "baseline" else specialized_dirs[domain]
        branch_dir = RUN_DIR / domain_dirs[domain] / branch
        stage3_arch = json.loads((branch_dir / "stage3_normalized_specter/architecture_compatibility_report.json").read_text()) if (branch_dir / "stage3_normalized_specter/architecture_compatibility_report.json").exists() else {}
        stage3_train = json.loads((branch_dir / "stage3_normalized_specter/stage3_trainability_report.json").read_text()) if (branch_dir / "stage3_normalized_specter/stage3_trainability_report.json").exists() else {}
        stage3_data = json.loads((branch_dir / "stage3_normalized_specter/domain_dataset_report.json").read_text()) if (branch_dir / "stage3_normalized_specter/domain_dataset_report.json").exists() else {}
        stage4_arch = json.loads((branch_dir / "corrected_stage4_normalized_specter/stage4_architecture_compatibility_report.json").read_text()) if (branch_dir / "corrected_stage4_normalized_specter/stage4_architecture_compatibility_report.json").exists() else {}
        selected_arch = stage3_arch.get("selected_ae_checkpoint_architecture", {})
        instantiated = stage3_arch.get("instantiated_stage3_architecture", {})
        fps = stage3_data.get("split_fingerprints", {})
        stage4_domain = stage4_arch.get("domain_filter_report", {})
        stage4_fps = {name: stage4_domain.get(name, {}).get("fingerprint", "") for name in ["train", "val", "test"]}
        stage4_domain_filter_passed = bool(stage4_domain) and all(stage4_fps.get(name) == fps.get(name) for name in ["train", "val", "test"])
        pair_fingerprints.setdefault(domain, []).append(fps)
        audit_rows.append({
            "run_name": run["run"],
            "domain": domain,
            "baseline_or_specialized": run["type"],
            "AE checkpoint": run.get("ae_checkpoint_path", ""),
            "AE base channels": selected_arch.get("base_channels", ""),
            "Stage 3 base channels": instantiated.get("base_channels", ""),
            "number of blocks": instantiated.get("num_blocks", ""),
            "output dimension": instantiated.get("out_dim", ""),
            "norm": instantiated.get("norm", ""),
            "pooling": instantiated.get("pooling", ""),
            "dropout": instantiated.get("dropout", ""),
            "encoder architecture": instantiated.get("encoder_arch", ""),
            "strict encoder load passed": stage3_arch.get("strict_load_success", ""),
            "CNN encoder trainable": stage3_train.get("cnn_encoder_trainable", ""),
            "text projection pretrained": stage3_train.get("text_projection_pretrained", ""),
            "text projection trainable": stage3_train.get("text_projection_trainable", ""),
            "symmetric InfoNCE confirmed": stage3_train.get("loss", "") == "symmetric InfoNCE",
            "train source counts": json.dumps(stage3_data.get("source_value_counts", {}).get("train", {}), sort_keys=True),
            "validation source counts": json.dumps(stage3_data.get("source_value_counts", {}).get("val", {}), sort_keys=True),
            "test source counts": json.dumps(stage3_data.get("source_value_counts", {}).get("test", {}), sort_keys=True),
            "train fingerprint": fps.get("train", ""),
            "validation fingerprint": fps.get("val", ""),
            "test fingerprint": fps.get("test", ""),
            "matching-pair fingerprints equal": "pending",
            "Stage 4 decoder strict load passed": stage4_arch.get("strict_autoencoder_load", "") == "passed",
            "Stage 4 domain filter passed": stage4_domain_filter_passed,
            "warnings": json.dumps({"stage3_arch": stage3_arch.get("unexpected_differences", {}), "stage3_train": stage3_train.get("failures", [])}, sort_keys=True),
            "status": "passed" if stage3_arch.get("final_compatibility_status") == "passed" and stage3_train.get("status") == "passed" else "incomplete_or_failed",
        })
    for row in audit_rows:
        fps_group = pair_fingerprints.get(row["domain"], [])
        row["matching-pair fingerprints equal"] = len(fps_group) == 2 and fps_group[0] == fps_group[1]
    if audit_rows:
        write_table(Path(paths["final"]) / "architecture_and_data_audit.csv", audit_rows)
except Exception as exc:
    print("WARNING: could not write architecture_and_data_audit.csv:", exc)

for s in stage_status:
    print(f"{s['stage']}: {s['status']}")
print("Final comparison:", Path(paths["final"]))


# 6a result packaging and comparison skeletons
summary_root = RUN_DIR / "03_all_domain_summary"
cache_audit_dir = RUN_DIR / "00_normalized_text_cache_audit"
stage3_summary_dir = RUN_DIR / "01_stage3_normalized"
stage4_summary_dir = RUN_DIR / "02_corrected_stage4_normalized"
for d in [summary_root, cache_audit_dir, stage3_summary_dir, stage4_summary_dir]:
    d.mkdir(parents=True, exist_ok=True)
for src in [NORMALIZED_PATHS.get("metadata"), NORMALIZED_PATHS.get("validation"), NORMALIZED_PATHS.get("index")]:
    try:
        if src and Path(src).exists():
            shutil.copy2(src, cache_audit_dir / Path(src).name)
    except Exception as exc:
        print("Cache audit copy skipped:", exc)

def read_json_if_exists(path: Path) -> dict:
    try:
        return json.loads(path.read_text()) if path.exists() else {}
    except Exception:
        return {}

stage3_rows = []
stage4_rows = []
legacy_stage4_rows = []
try:
    manifest = json.loads(Path(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST).read_text()) if STAGE2_STAGE3_STAGE4_INPUT_MANIFEST else {"six_stage2_stage3_stage4_runs": []}
    domain_dirs = {"pubmed": "01_pubmed", "nilearn": "02_nilearn", "neurovault": "03_neurovault"}
    specialized_dirs = {"pubmed": "specialized_mixed_to_pubmed", "nilearn": "specialized_mixed_to_nilearn", "neurovault": "specialized_mixed_to_neurovault"}
    for run in manifest["six_stage2_stage3_stage4_runs"]:
        domain = run["domain"]
        branch = "baseline_mixed_stage1a" if run["type"] == "baseline" else specialized_dirs[domain]
        branch_dir = RUN_DIR / domain_dirs[domain] / branch
        stage3_dir = branch_dir / "stage3_normalized_specter"
        test_metrics = read_json_if_exists(stage3_dir / "test_metrics.json")
        comparison = read_json_if_exists(stage3_dir / "comparison_row.json")
        stage3_rows.append({
            "run": f"{run['run']}_normalized_specter",
            "domain": domain,
            "branch": run["type"],
            "normalized_input_auc": comparison.get("paper_recall_curve_auc", test_metrics.get("paper_recall_curve_auc", "")),
            "text_to_brain_auc": test_metrics.get("t2i_normalized_k_recall_curve_auc", ""),
            "brain_to_text_auc": test_metrics.get("i2t_normalized_k_recall_curve_auc", ""),
            "checkpoint": str(stage3_dir / "checkpoints/best_val_normalized_recall_auc.pt"),
            "normalized_cache_sha256": sha256_file(NORMALIZED_SPECTER_CACHE),
            "architecture_equality": "expected_equal_except_text_preprocessing",
            "hyperparameter_equality": "expected_equal_except_text_preprocessing",
        })
        stage4_dir = branch_dir / "corrected_stage4_normalized_specter"
        eval_rows_path = stage4_dir / "generation_eval_metrics.json"
        eval_rows = json.loads(eval_rows_path.read_text()) if eval_rows_path.exists() else []
        all_row = next((r for r in eval_rows if r.get("source") == "all"), {})
        base_run = run["run"]
        corrected_auc = all_row.get("generation_mean_normalized_auc", "")
        legacy_auc = LEGACY_STAGE4_BASELINES.get(base_run, "")
        stage4_rows.append({
            "run": f"{base_run}_normalized_stage4",
            "domain": domain,
            "branch": run["type"],
            "corrected_generation_auc": corrected_auc,
            "text_to_generated_brain_auc": all_row.get("generation_text_to_brain_normalized_auc", ""),
            "generated_brain_to_text_auc": all_row.get("generation_brain_to_text_normalized_auc", ""),
            "matched_contrastive_cosine": all_row.get("generation_matched_contrastive_cosine", ""),
            "shuffled_null_contrastive_cosine": all_row.get("generation_shuffled_contrastive_cosine", ""),
            "spatial_corr": all_row.get("spatial_corr", ""),
            "top1_dice": all_row.get("top1_dice", ""),
            "top5_dice": all_row.get("top5_dice", ""),
            "top10_dice": all_row.get("top10_dice", ""),
            "mse": all_row.get("mse", ""),
            "mae": all_row.get("mae", ""),
            "foreground_mse": all_row.get("foreground_mse", ""),
            "pred_nonzero_fraction": all_row.get("pred_nonzero_fraction", ""),
            "target_nonzero_fraction": all_row.get("target_nonzero_fraction", ""),
            "primary_checkpoint": str(stage4_dir / "checkpoints/best_val_generation_normalized_auc.pt"),
        })
        legacy_stage4_rows.append({
            "run": f"{base_run}_normalized_stage4",
            "legacy_generation_auc": legacy_auc,
            "corrected_generation_auc": corrected_auc,
            "difference": (float(corrected_auc) - float(legacy_auc)) if corrected_auc != "" and legacy_auc != "" else "",
            "exact_checkpoint_name": str(stage4_dir / "checkpoints/best_val_generation_normalized_auc.pt"),
            "split_fingerprints": json.dumps(read_json_if_exists(stage4_dir / "stage4_domain_dataset_report.json"), sort_keys=True),
        })
except Exception as exc:
    print("WARNING: summary packaging incomplete:", exc)

write_table(summary_root / "stage3_normalized_all_runs.csv", stage3_rows)
write_table(summary_root / "stage4_corrected_normalized_all_runs.csv", stage4_rows)
write_table(summary_root / "stage1_stage3_stage4_normalized_summary.csv", stage3_rows + stage4_rows)
write_table(stage3_summary_dir / "normalized_vs_legacy_stage3.csv", stage3_rows)
write_table(stage4_summary_dir / "normalized_corrected_vs_legacy_stage4.csv", legacy_stage4_rows)
README = f"""# 6a normalized-SPECTER2 summary

1. Did unit-normalized SPECTER2 improve Stage 3 normalized recall AUC? See `03_all_domain_summary/stage3_normalized_all_runs.csv` and `01_stage3_normalized/normalized_vs_legacy_stage3.csv` after the six runs complete.
2. Did the normalized-input Stage 3 results outperform the saved legacy results? The comparison table is prepared in `01_stage3_normalized/normalized_vs_legacy_stage3.csv`; populate legacy Stage 3 baselines from the fixed completed baseline output root if they are not already discoverable.
3. Did the corrected independent generative projector improve Stage 4 semantic AUC? See `02_corrected_stage4_normalized/normalized_corrected_vs_legacy_stage4.csv` using the fixed legacy AUC baselines.
4. Did it reduce diffuse predictions and improve spatial fidelity? Inspect `pred_nonzero_fraction`, Dice, spatial correlation, MSE, MAE, and foreground MSE in `03_all_domain_summary/stage4_corrected_normalized_all_runs.csv`.
5. Which AE initialization is best for each domain? Compare baseline mixed Stage 1A versus specialized Stage 1B rows within each domain in the Stage 3 and Stage 4 summary CSVs.
6. Should normalized SPECTER2 become the default for future Stage 3 and Stage 4 experiments? Decide from the paired Stage 3 AUC and corrected Stage 4 semantic/spatial tables once all six branches are complete.

Normalized cache: `{NORMALIZED_SPECTER_CACHE}`
Cache SHA-256: `{sha256_file(NORMALIZED_SPECTER_CACHE)}`
Preprocessing: `{TEXT_EMBEDDING_PREPROCESSING}`
"""
(summary_root / "README_WHAT_TO_LOOK_AT.md").write_text(README)
print("6a packaged outputs:", RUN_DIR)
print("Summary:", summary_root)
